# 07 — Instrument closure: pin the probe, then re-sweep the realized grid

**BLIND-SAFE up to section 9. Run this BEFORE notebook 06.** Sections 1-8 score only the
**shape anchor** (the §5 gate factor, orthogonal to H1-H4) and the raw-pixel reference.
No targeted-factor value is read anywhere in this notebook.

## Why this notebook exists

The first calibration run (`results/calibration/calibration_shapes3d.raw.json`) is **void**.
It failed on three counts:

1. **A9 — budget confound.** The probe budget was fixed in *epochs*, so probe-train size and
   optimizer budget moved together (100 steps at n=2000 vs 1000 at n=40000). Headroom that
   appeared at small n could not be told apart from a probe that never converged.
2. **Degenerate extrapolation regime.** `make_value_holdout_splits` withholds whole *values*
   of the anchor. For a class label that leaves probe-test holding only unseen classes, so
   accuracy is 0 by construction: all 8 extrapolation cells returned the -1/3 zero-accuracy
   floor. That arm measured nothing.
3. **Selector artifact.** `recommended_config` filtered on headroom alone and sorted
   extrapolation first, so the regime that fails by construction — and is therefore never
   saturated — won automatically. Its verdict was a constant of the code.

All three are repaired in `src/`. This notebook re-runs the calibration under the repaired
instrument, pins the probe config on a criterion that both A8 §c and A8 §e must clear, and then
re-sweeps the three realized cells at that config.

## The re-sweep is mandatory regardless

The three existing stacks are stale against the current instrument: they carry four arrays
(`trained`, `random`, `perm`, `projector`) and are missing `random_projector` — the A8 §d
matched random-projector floor that H4 needs — and their `meta.json` has no `encoder_ckpts`
provenance. They must be re-swept whatever the calibration says, so pinning a probe config
first costs no extra compute.

## The decision rule, pre-committed here while blind

Read the branch off section 8 and follow it. Committing to this *before* seeing the numbers is
the point; choosing after would repeat the first run's error.

| Branch | Condition | Action |
|---|---|---|
| **A** | some `(regime=interpolation, n, steps)` clears §c (top-rung random floor < 0.90) **and** §e (\|gap\| <= 0.02) on **both** datasets | Pin it. Re-sweep (§9). Then unblind with notebook 06. |
| **B** | only `regime=composition` clears both | Stop. File amendment A10 changing the registered probe-test split, and extend `run_sweep` with the composition regime, before any sweep. |
| **C** | something clears §c but nothing clears §e | Do **not** pin on headroom alone — that is exactly the first run's error. Raise `--probe-steps` and re-run §6-§7. If the gap holds, invoke A9 consequence (2): report the linear rung as optimizer-limited and co-report every `Delta_G` against the closed-form rung. |
| **D** | nothing clears §c at any n, steps or regime | Saturation is a property of these datasets, not of the probe. Keep n=40000, run the confirmatory layer with A6(a)'s saturation gate active, and report the saturation itself as the finding plus the un-gated flip variants as diagnostics. Write it up as a pre-registered limitation, not a post-hoc discovery. |

**Setup:** Accelerator `GPU T4 x2`, Internet **On**.
**Outputs:** `results/calibration/calibration_{shapes3d,dsprites}.json`, then
`results/probes/{color,control,position}_strong/{stacks.npz,meta.json}`.

## 1. Verify the GPU(s)

In [ ]:
!nvidia-smi

## 2. Clone the repo
Onto `/kaggle/working` (persists across restarts within a session).

In [ ]:
import os

REPO_URL = "https://github.com/chinesegorilla99/probe-capacity-invariance.git"
REPO_DIR = "/kaggle/working/probe-capacity-invariance"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}

## 3. Install dependencies
Without disturbing Kaggle's preinstalled, CUDA-matched `torch`/`torchvision`.

In [ ]:
!pip install -q -e . --no-deps
!pip install -q h5py

In [ ]:
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "| device count:", torch.cuda.device_count())

## 4. Datasets + image cache
`--build-cache` decompresses once into an uncompressed memmap the loaders mmap. Idempotent.

In [ ]:
!cd /kaggle/working/probe-capacity-invariance && python -m src.data.shapes3d --download --build-cache
!cd /kaggle/working/probe-capacity-invariance && python -m src.data.dsprites --download --build-cache

In [ ]:
# Restore prior probe/calibration OUTPUTS and encoder checkpoints so a timed-out
# session continues instead of recomputing.
import shutil
from pathlib import Path

REPO = Path("/kaggle/working/probe-capacity-invariance"); INPUT = Path("/kaggle/input")
restored = 0
for src in list(INPUT.glob("*/results")) + list(INPUT.glob("*/probe-capacity-invariance/results")):
    for f in src.rglob("*"):
        if f.is_file() and f.suffix in (".npz", ".json", ".jsonl", ".pt"):
            dst = REPO / "results" / f.relative_to(src)
            if not dst.exists():
                dst.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(f, dst); restored += 1
print(f"restored {restored} prior result files")

## 5. Preflight — is the instrument actually repaired?

Fails loudly rather than burning a session on the old code. Also reports what the existing probe
stacks are missing, which is why section 9 has to run either way.

In [ ]:
import inspect, json
from pathlib import Path
import numpy as np

from src.probes import ladder, instrument_calibration
from src.data import splits
from src.eval import metrics

checks = [
    ("A9 step budget pinned", getattr(ladder, "DEFAULT_STEPS", None) == 1000,
     f"ladder.DEFAULT_STEPS={getattr(ladder, 'DEFAULT_STEPS', None)}"),
    ("fit_rung accepts steps", "steps" in inspect.signature(ladder.fit_rung).parameters, ""),
    ("compositional split available", hasattr(splits, "make_combination_holdout_splits"), ""),
]
src_cal = inspect.getsource(instrument_calibration.run)
checks += [
    ("categorical extrapolation guarded", 'anchor.kind == "categorical"' in src_cal, ""),
    ("selector requires BOTH §c and §e", 'capacity_axis_flag"]' in src_cal, ""),
    ("closed-form solver max_iter raised",
     "max_iter=5000" in inspect.getsource(metrics.linear_recoverability_categorical), ""),
]

width = max(len(n) for n, _, _ in checks)
for name, ok, detail in checks:
    print(f"{'PASS' if ok else 'FAIL'}  {name:<{width}}  {detail}")
bad = [n for n, ok, _ in checks if not ok]
assert not bad, f"instrument not repaired: {bad} — `git pull` and re-run this cell"

print("\nExisting probe stacks (why section 9 is mandatory):")
for cell in ("color_strong", "control_strong", "position_strong"):
    d = Path("results/probes") / cell
    if not (d / "stacks.npz").exists():
        print(f"  {cell:16s} absent"); continue
    z, m = np.load(d / "stacks.npz"), json.loads((d / "meta.json").read_text())
    missing = [k for k in ("random_projector",) if k not in z.files]
    print(f"  {cell:16s} n={m.get('probe_train_size')} arrays={len(z.files)} "
          f"missing={missing or 'none'} ckpt_provenance={'yes' if m.get('encoder_ckpts') else 'NO'}")

## 6. Shapes3D calibration (repaired)

Two step budgets, so the A8 §e gap can be read as a function of optimizer budget rather than
assumed away: at n=40000 the *old* run already had 1000 steps and still showed a +0.065 gap, so
1000 alone is not known to be enough. `orientation` is the compositional partner because its arm
is excluded from the realized grid (A5), so the split touches no targeted factor.

In [ ]:
!cd /kaggle/working/probe-capacity-invariance && python -m src.probes.instrument_calibration \
    --dataset shapes3d --probe-train-sizes 2000 5000 10000 40000 \
    --probe-steps 1000 4000 --partner-factor orientation --n-holdout-partner 3 \
    --random-seed 0 1 2 --device cuda --num-workers 2 --out-root results/calibration

## 7. dSprites calibration (the position arm's dataset)

Partner is `scale`: `pos_x`/`pos_y` are the position arm's targeted factors, and dSprites
`orientation` is diagnostic-only per A3.

In [ ]:
!cd /kaggle/working/probe-capacity-invariance && python -m src.probes.instrument_calibration \
    --dataset dsprites --probe-train-sizes 2000 5000 10000 40000 \
    --probe-steps 1000 4000 --partner-factor scale --n-holdout-partner 2 \
    --random-seed 0 1 2 --device cuda --num-workers 2 --out-root results/calibration

## 8. The decision

A config is pinnable only if it clears **both** gates on **both** datasets:

* **A8 §c** — top-rung random-encoder floor < 0.90, so `G` has range to move in.
* **A8 §e** — the ladder's Adam linear rung agrees with the convex solver to within 0.02, so
  `Delta_G` is a capacity difference and not an optimization difference.

Cells that clear §c but fail §e are printed separately: they are the trap the first run fell into.

In [ ]:
import json
from pathlib import Path

TOL_REGIMES = ("interpolation", "composition")   # preference order
DATASETS = ("shapes3d", "dsprites")

runs = {}
for ds in DATASETS:
    p = Path(f"results/calibration/calibration_{ds}.json")
    if not p.exists():
        print(f"{ds}: NOT RUN"); continue
    runs[ds] = json.loads(p.read_text())

def clears_c(r): return not r["saturated_at_top"]
def clears_e(r): return not r["capacity_axis_flag"]

for ds, r in runs.items():
    print(f"\n=== {ds} ===  partner={r.get('composition_partner')}  tol={r.get('gap_tolerance')}")
    print(f"{'regime':15s}{'n':>7s}{'steps':>7s}{'role':>8s}{'top floor':>11s}"
          f"{'headroom':>10s}{'lin gap':>9s}  gates")
    for row in r["results"]:
        gates = ("§c " if clears_c(row) else "-- ") + ("§e" if clears_e(row) else "--")
        pin = (row["encoder_role"] == "random" and clears_c(row) and clears_e(row))
        print(f"{row['regime']:15s}{row['probe_train_size']:>7d}{row['probe_steps']:>7d}"
              f"{row['encoder_role']:>8s}{row['top_rung_floor']:>11.4f}"
              f"{row['headroom_at_top']:>10.4f}{row['linear_rung_gap']:>+9.4f}  {gates}"
              + ("  <-- PINNABLE" if pin else ""))

# A config must clear both gates on the random encoder of EVERY dataset, so the two arms stay
# comparable; per-dataset pins would confound the arm contrast with the probe config.
pinnable = None
if len(runs) == len(DATASETS):
    per_ds = [
        {(r["regime"], r["probe_train_size"], r["probe_steps"])
         for r in run["results"]
         if r["encoder_role"] == "random" and clears_c(r) and clears_e(r)}
        for run in runs.values()
    ]
    common = {k for k in set.intersection(*per_ds)} if per_ds else set()
    common = {k for k in common if k[0] in TOL_REGIMES}
    if common:
        pinnable = min(common, key=lambda k: (TOL_REGIMES.index(k[0]), -k[1], k[2]))

headroom_only = sorted({
    (r["regime"], r["probe_train_size"], r["probe_steps"])
    for run in runs.values() for r in run["results"]
    if r["encoder_role"] == "random" and clears_c(r) and not clears_e(r)
})

print("\n" + "=" * 78)
if pinnable:
    REGIME, PROBE_TRAIN, PROBE_STEPS = pinnable
    BRANCH = "A" if REGIME == "interpolation" else "B"
    print(f"BRANCH {BRANCH}: pinnable on both datasets -> regime={REGIME} "
          f"probe_train={PROBE_TRAIN} probe_steps={PROBE_STEPS}")
    if BRANCH == "B":
        print("STOP. Branch B needs amendment A10 (the registered probe-test split becomes\n"
              "compositional) and a --regime path in run_sweep before section 9 may run.")
else:
    REGIME = PROBE_TRAIN = PROBE_STEPS = None
    BRANCH = "C" if headroom_only else "D"
    print(f"BRANCH {BRANCH}: nothing clears both gates on both datasets.")
    if BRANCH == "C":
        print("Cells with headroom but an optimizer-limited linear rung (do NOT pin these):")
        for k in headroom_only:
            print(f"   regime={k[0]} n={k[1]} steps={k[2]}")
        print("Raise --probe-steps in sections 6-7 and re-run, or invoke A9 consequence (2).")
    else:
        print("Saturation is a property of the datasets, not the probe. Keep n=40000 and\n"
              "report under the A6(a) saturation gate — see branch D in the header.")
print("=" * 78)

## 9. Re-sweep the three realized cells at the pinned config

**This is the last blind-safe section.** The sweep fits every factor, but nothing here prints a
targeted-factor value — it writes `stacks.npz` and reports only the shape-gate summary. Read
`results/hypotheses/` (notebook 06) only after this completes.

`control_strong` is gate-exempt by design (A2): a minimal-augmentation encoder is *expected* to
sit at the random floor, so 0/12 passing is the intended datum, not a failure.

In [ ]:
from pathlib import Path

assert BRANCH == "A", (
    f"branch {BRANCH}: do not sweep. Section 8 printed what to do instead — sweeping here "
    "would pin a probe config the calibration did not license."
)

# run_sweep only WARNS when it gets too few encoders, so an unresolved glob would
# silently sweep zero trained encoders and write a stack that looks valid. Fail here.
CELLS = {
    "color_strong": "results/encoders/color_strong_seed*/backbone.pt",
    "control_strong": "results/encoders/control_strong_seed*/backbone.pt",
    "position_strong": "results/encoders/position_strong_seed*/backbone.pt",
}
for cell, pattern in CELLS.items():
    found = sorted(Path().glob(pattern))
    print(f"{cell:16s} {len(found):>2d} checkpoints")
    assert len(found) == 12, (
        f"{cell}: '{pattern}' resolved {len(found)} checkpoints, expected 12. The restored "
        "input layout differs from the training layout — fix the glob before sweeping."
    )

SEEDS = "0 1 2 3 4 5 6 7 8 9 10 11"
print(f"\nsweeping at probe_train={PROBE_TRAIN} probe_steps={PROBE_STEPS} regime={REGIME}")

In [ ]:
!cd /kaggle/working/probe-capacity-invariance && python -m src.probes.run_sweep \
    --config configs/probe/ladder.yaml \
    --dataset shapes3d --condition color --strength strong \
    --encoders results/encoders/color_strong_seed*/backbone.pt \
    --random-seed {SEEDS} --subsample {PROBE_TRAIN} --probe-steps {PROBE_STEPS} \
    --device cuda --num-workers 2 --resume --out-root results/probes

In [ ]:
!cd /kaggle/working/probe-capacity-invariance && python -m src.probes.run_sweep \
    --config configs/probe/ladder.yaml \
    --dataset shapes3d --condition control --strength strong \
    --encoders results/encoders/control_strong_seed*/backbone.pt \
    --random-seed {SEEDS} --subsample {PROBE_TRAIN} --probe-steps {PROBE_STEPS} \
    --device cuda --num-workers 2 --resume --out-root results/probes

In [ ]:
!cd /kaggle/working/probe-capacity-invariance && python -m src.probes.run_sweep \
    --config configs/probe/ladder.yaml \
    --dataset dsprites --condition position --strength strong \
    --encoders results/encoders/position_strong_seed*/backbone.pt \
    --random-seed {SEEDS} --subsample {PROBE_TRAIN} --probe-steps {PROBE_STEPS} \
    --device cuda --num-workers 2 --resume --out-root results/probes

## 10. Verify the contract artifacts

Every cell must carry the A8 §d random-projector floor, checkpoint provenance, and the pinned
probe config. A cell that fails here is not usable by notebook 06.

In [ ]:
import json
import numpy as np
from pathlib import Path

ok = True
for cell in ("color_strong", "control_strong", "position_strong"):
    d = Path("results/probes") / cell
    if not (d / "stacks.npz").exists():
        print(f"{cell:16s} NOT SWEPT"); ok = False; continue
    z = np.load(d / "stacks.npz"); m = json.loads((d / "meta.json").read_text())
    problems = []
    if "random_projector" not in z.files: problems.append("no random_projector (A8 §d)")
    if not m.get("encoder_ckpts"): problems.append("no checkpoint provenance")
    if m.get("probe_train_size") != PROBE_TRAIN: problems.append(f"n={m.get('probe_train_size')}")
    if m.get("probe_steps") != PROBE_STEPS: problems.append(f"steps={m.get('probe_steps')}")
    g = m["quality_gate"]
    print(f"{cell:16s} {'OK  ' if not problems else 'BAD '} "
          f"seeds={z['trained'].shape[0]} gate={g['n_passed']}/{g['n_encoders']} "
          f"{'; '.join(problems)}")
    ok &= not problems
print("\nready for notebook 06" if ok else "\nNOT ready — fix the above before unblinding")

## 11. Persist for the next session
`/kaggle/working` is the notebook output. Click **Save Version** when this finishes, then
**Add Input -> this output** on the next run to resume.

In [ ]:
import shutil
from pathlib import Path
src = Path("/kaggle/working/probe-capacity-invariance/results"); dst = Path("/kaggle/working/results")
shutil.rmtree(dst, ignore_errors=True); shutil.copytree(src, dst)
print(f"persisted {sum(1 for _ in dst.rglob('*') if _.is_file())} files "
      f"-> click 'Save Version' now")